# Train SASRec

In [1]:
import numpy as np

# For NumPy 2.0 compatibility with RecBole 1.2
np.float_ = np.float64
np.int_ = np.int64
np.complex_ = np.complex128
np.unicode_ = np.str_

In [2]:
from typing import Any
import torch
import pandas as pd
from recbole.config import Config
from recbole.data.dataloader import FullSortEvalDataLoader, AbstractDataLoader
from recbole.data import create_dataset, data_preparation
from recbole.model.sequential_recommender import SASRec
from recbole.trainer import Trainer
from recbole.utils import init_seed, init_logger

In [ ]:
# --- Config ---
# Assume we have `*.train.inter`, `*.valid.inter`, `*.test.inter`
DATASET_NAME: str = "target" 
DATA_DIR: str = "../data"
SEED = 67
DEVICE = "mps" # Other options: "cpu", "cuda"

# Sequential recommendation config
MAX_ITEM_LIST_LENGTH: int = 50

## Create dataset

In [ ]:
config_dict: dict[str, Any] = {
    "data_path": DATA_DIR,
    "dataset": DATASET_NAME,
    "USER_ID_FIELD": "user_id",
    "ITEM_ID_FIELD": "item_id",
    "RATING_FIELD": "rating",
    "benchmark_filename": ["train", "valid", "test"],
    "load_col": {
        "inter": ["user_id", "item_id", "rating", "item_id_list"],
        "user": ["user_id", "category"] 
    },
    # This is needed to tell RecBole that `item_id_list` is an alias of `item_id` for sequential recommendation.
    "alias_of_item_id": ["item_id_list"],
    "MAX_ITEM_LIST_LENGTH": MAX_ITEM_LIST_LENGTH,
    "epochs": 100,
    "train_batch_size": 1024,
    "eval_batch_size": 1024,
    # Not relevant for SASRec
    "train_neg_sample_args": None,
    "eval_args": {
        # Split is already determined by the `benchmark filename` as separate `.inter` files
        "split": None, 
        "order": "TO",
        "mode": "full",
    },
    "metrics": ["NDCG", "Recall", "MRR"],
    "valid_metric": "NDCG@10",
    "seed": SEED,
}

config: Config = Config(model="SASRec", config_dict=config_dict)
config.final_config_dict["device"] = torch.device(DEVICE)

init_logger(config)
init_seed(SEED, reproducibility=True)

Configuring MPS (Apple Silicon GPU) for training.


In [5]:
dataset = create_dataset(config)
train_data, valid_data, test_data = data_preparation(config, dataset)

/Users/nginyc/repos/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/data/dataset/dataset.py:501: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[field].fillna(value="", inplace=True)
/Users/nginyc/repos/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/data/dataset/dataset.py:501: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because

## Train SASRec

In [6]:
model: SASRec = SASRec(config, train_data.dataset).to(config["device"])
trainer: Trainer = Trainer(config, model)

best_valid_score: float
best_valid_result: dict[str, float]
best_valid_score, best_valid_result = trainer.fit(train_data, valid_data)

/Users/nginyc/repos/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/trainer/trainer.py:235: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler(enabled=self.enable_scaler)
18 Jun 13:35    INFO  epoch 0 training [time: 18.34s, train loss: 859.6357]
18 Jun 13:35    INFO  epoch 0 evaluating [time: 0.91s, valid_score: 0.004300]
18 Jun 13:35    INFO  valid result: 
ndcg@10 : 0.0043    recall@10 : 0.005    mrr@10 : 0.0041
18 Jun 13:35    INFO  Saving current: saved/SASRec-Jun-18-2026_13-35-14.pth
18 Jun 13:35    INFO  epoch 1 training [time: 18.97s, train loss: 838.6486]
18 Jun 13:35    INFO  epoch 1 evaluating [time: 0.69s, valid_score: 0.001600]
18 Jun 13:35    INFO  valid result: 
ndcg@10 : 0.0016    recall@10 : 0.0032    mrr@10 : 0.0012
18 Jun 13:36    INFO  epoch 2 training [time: 16.43s, train loss: 827.0067]
18 Jun 13:36    INFO  epoch 2 evaluating [time: 0.61s, vali

In [10]:
print(f"\nBest valid score: {best_valid_score:.4f}")
print("Best valid result:")
for metric, score in best_valid_result.items():
    print(f"  {metric}: {score:.4f}")


Best valid score: 0.0043
Best valid result:
  ndcg@10: 0.0043
  recall@10: 0.0050
  mrr@10: 0.0041


## Evaluate on test set

In [11]:
test_result: dict[str, float] = trainer.evaluate(test_data)

print("Test results (Overall):")
for metric, value in test_result.items():
    print(f"  {metric}: {value:.4f}")

18 Jun 13:39    INFO  Loading model structure and parameters from saved/SASRec-Jun-18-2026_13-35-14.pth


Test results (Overall):
  ndcg@10: 0.0027
  recall@10: 0.0035
  mrr@10: 0.0024


In [12]:
def evaluate_on_subset(
    data: AbstractDataLoader,
    mask: np.ndarray,
    label: str
):
    '''
    Evaluate the model on a subset of interactions defined by `mask`.
    '''
    inter_feat = data.dataset.inter_feat
    cat_ds = data.dataset.copy(inter_feat[mask])
    cat_dl = FullSortEvalDataLoader(config, cat_ds, sampler=data._sampler)
    results = trainer.evaluate(cat_dl)
    print(f"\nEvaluation ({label})")
    print(f'-' * 20)
    print(f"  Interactions: {mask.sum()}")
    for metric, val in results.items():
        print(f"  {metric}: {val:.4f}")

# Map integer categories to labels
tok = dataset.field2token_id["category"]
CAT_LABELS = {tok["0"]: "warm", tok["1"]: "cold", tok["2"]: "new"}

uid_to_cat = dict(zip(
    dataset.user_feat[dataset.uid_field].numpy(),
    dataset.user_feat["category"].numpy(),
))

uid_array = test_data.dataset.inter_feat[dataset.uid_field].numpy()

for cat_id, cat_label in CAT_LABELS.items():
    cat_uids = {uid for uid, c in uid_to_cat.items() if c == cat_id}
    mask = np.isin(uid_array, list(cat_uids))
    if not mask.any():
        print(f"\n  {cat_label}: no users in test set — skipping")
        continue

    evaluate_on_subset(test_data, mask, cat_label)

18 Jun 13:39    INFO  Loading model structure and parameters from saved/SASRec-Jun-18-2026_13-35-14.pth



Evaluation (warm)
--------------------
  Interactions: 4495
  ndcg@10: 0.0033
  recall@10: 0.0040
  mrr@10: 0.0032


18 Jun 13:39    INFO  Loading model structure and parameters from saved/SASRec-Jun-18-2026_13-35-14.pth



Evaluation (cold)
--------------------
  Interactions: 2678
  ndcg@10: 0.0014
  recall@10: 0.0019
  mrr@10: 0.0013


18 Jun 13:39    INFO  Loading model structure and parameters from saved/SASRec-Jun-18-2026_13-35-14.pth



Evaluation (new)
--------------------
  Interactions: 2826
  ndcg@10: 0.0028
  recall@10: 0.0042
  mrr@10: 0.0024
